# CS-546 Final Project
## 
## Authors: Tobi Kohn & Trent VanHawkins

### Introduction

Pathogens displaying Long-Distance Dispersal (LDD) pose a substantial threat to global food security and agronomics. Existing agricultural managemenet practices often include treatments such as fungicide, herbicide, or pesticide; however,  airborne pathogens can readily invade across abritrary jurisdictional boundaries and propogate new infections where management practices differ. Anually, these diseases pose an estimated $220 Billion in harvest loss [CITE]. The Willamette Valley in particular is a global leader in the production of Hops (*Humulus lupulus*) and Powdery Mildew (*Podosphaera macularis*) is a regionally important diesease which can lead to total crop loss if not managed appropriately [CITE]. The disease appeared in the Willamette Valley for the first time in 1998, and has occured in every subsequent year up until the present. As such, the hop powdery mildew pathosystem has become one of particular concern for growers and managers within the region.

Models of disease spread are a classical focus of pathology research. In recent years, however, these classical, mechanistic models have been placed into the context of spatially explicit networks, which use stochastic methods to learn network structures across discrete time transitions [CITE]. Letting $t$ denote time, we define a *discrete time transition* as the transiton from the disease state at $t - 1$ to time $t$. Then, we can treat classical models of auto-infection--how much disease spread at a particular unit is due to its previous infection locally-- and dispersal--how much disease is due to infection from other sourcses-- as covariate kernels in a likelihood based framework. Thus, conditioning on the disease state at $t-1$, we can try to learn the parameters associated with each kernel to predict the disease state at time $t$.

From 2014 to 2017, Hop growers in the Willamette Valley partook in an annual sentinal monitoring program for powdery mildew. Farms were surveyed monthly for the number of infected plants in each yard, the experimenatl unit, from May to July. In the present study, we aim to recreate the network structure from these disease data using the framework proposed by [CITE] and [CITE]. An interesting feature of the proposed method is that the resulting adjasency matrix must account for the restriction illustrated by Figure 1. 

!["Figure 1: A simple graph denoting constraints of disease spread within the resulting network."](figures/strain_contraints.png)

Briefly, each yard can grow one hop cultivar in each season -- R6 or non-R6. R6 cultivars have been genetically modified for resistance against hop powdery mildew. As often occurs, though, a strain of the disease has mutated to overcome the R6 resistance. This mutated strain is termed a "V6" pathogen, thus the disease strain can be "V6" or "non-V6". The initial disease strain was sampled at each yard at first detection each season, allowing for three different possibilities: an R6 yard with initial strain V6, a non-R6 yard with initial strain V6, or a non-R6 yard with initial strain non-V6. The case of V6 yards with initial strain non-V6 are not considered biologically meaningful (the presence of spores does not imply disease spread) and are ommitted for clarity. The restriction on our network is as follows: **non-R6 yards may only infect R6 yards if they have initial strain V6**. We elaborate on how this restriction is imposed in the methods section below. 

Clearly, this constrained, directed, and weighted adjacency matrix proposes a compelling opportunity to explore some of the analysis tools we have discussed in CS 546. In the present study, we propose to investigate the feasability and behavior of some basic tools of netowrk analysis including vertex-degree, and basic measures of vertex centrality. We then use these measures to create a descriptive profile of important yards for disease transimission within the Willamette Valley. 

### Methods
#### The Data
#### How we learned the network structure
#### How we estimated the edge-weights
#### Estimating source-strength (outward degree) and an investigation of its properties
#### Analysis of source-strength stratified by yard-type and grower characteristics (sprays & pruning)
#### 

### Results
#### Source Strength behaviors
#### Soruce stregnth stratified analysis
#### Centrality analysis

### Discussion
#### General summary of conclusions
#### Limitations
#### Future directions

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cairo
import contextily as ctx
import networkx as nx

# Read in the adjacency matrices from the CSV files
apr_may = pd.read_csv('../reports/networks/network_may_jun.csv', index_col=0, header=0)
may_jun = pd.read_csv('../reports/networks/network_may_jun.csv', index_col=0, header=0)
jun_jul = pd.read_csv('../reports/networks/network_jun_jul.csv', index_col=0, header=0)

#Read in the metadata
meta = pd.read_csv('../data/processed/clean_summary.csv', index_col=0, header=0)
meta["date"] = pd.to_datetime(meta["date"])

# Subset to 2017
meta_2017 = meta.loc[meta["year"] == 2017, ["field_id", "centroid_lat", "centroid_long", "initial_strain", "plant_type"]]
meta_2017.drop_duplicates(subset="field_id", keep="first", inplace=True)

#Write a function to build an igraph with metadata
def build_graph(adj_df, meta_df):
    # Build graph
    g = ig.Graph.Weighted_Adjacency(adj_df.to_numpy().transpose(), mode="directed", attr="weight", loops=False)
    
    # Add node names
    g.vs["name"] = adj_df.columns.tolist()
    
    # Add metadata attributes
    meta_indexed = meta_df.set_index("field_id")
    for col in meta_indexed.columns:
        g.vs[col] = [meta_indexed.loc[int(v["name"]), col] for v in g.vs]
    
    return g

apr_may_g = build_graph(apr_may, meta_2017)
may_jun_g = build_graph(may_jun, meta_2017)
jun_jul_g = build_graph(jun_jul, meta_2017)


In [ ]:
out_apr_may = apr_may_g.strength(mode = "out", weights="weight")
out_may_jun = may_jun_g.strength(mode = "out", weights="weight")
out_jun_jul = jun_jul_g.strength(mode = "out", weights="weight")

In [ ]:
def plot_network(g, title="", weight_percentile=75):
    
    # Filter edges below percentile threshold
    weights = [e["weight"] for e in g.es]
    threshold = np.percentile(weights, weight_percentile)
    g_filtered = g.copy()
    g_filtered.delete_edges([e.index for e in g_filtered.es if e["weight"] < threshold])

    # Remove isolated nodes (optional)
    g_filtered.delete_vertices([v.index for v in g_filtered.vs if g_filtered.degree(v.index) == 0])

    # Node properties
    pos = {v["name"]: (v["centroid_long"], v["centroid_lat"]) for v in g_filtered.vs}
    node_colors = ["crimson" if v["plant_type"] == "R6" else "steelblue" for v in g_filtered.vs]
    weighted_outdegree = g_filtered.strength(mode="out", weights="weight")
    # Normalise to a size range
    min_size, max_size = 100, 1000
    w_min, w_max = min(weighted_outdegree), max(weighted_outdegree)
    node_sizes = [min_size + (w - w_min) / (w_max - w_min + 1e-9) * (max_size - min_size) 
                for w in weighted_outdegree]

    # Edge properties
    edge_weights = [e["weight"] for e in g_filtered.es]
    w_min, w_max = min(edge_weights), max(edge_weights)
    edge_colors = [(0, 0, 0, (e["weight"] - w_min) / (w_max - w_min + 1e-9)) for e in g_filtered.es]  

    # Convert to networkx for plotting
    G = nx.DiGraph()
    for v in g_filtered.vs:
        G.add_node(v["name"])
    for e in g_filtered.es:
        G.add_edge(g_filtered.vs[e.source]["name"], g_filtered.vs[e.target]["name"], weight=e["weight"])

    fig, ax = plt.subplots(figsize=(12, 10))

    nx.draw_networkx_nodes(G, pos=pos, ax=ax, node_color=node_colors,
                           node_size=node_sizes, alpha=0.75)
    nx.draw_networkx_labels(G, pos=pos, ax=ax, font_size=6, font_color="white")
    nx.draw_networkx_edges(G, pos=pos, ax=ax, edge_color=edge_colors,
                           arrows=True, arrowsize=15, width=1.5,
                           connectionstyle="arc3,rad=0.1")  # slight curve to show directionality

    # Basemap
    ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.Voyager)

    # Legend
    from matplotlib.lines import Line2D
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson", markersize=10, label="R6"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=10, label="Non-R6")]
    ax.legend(handles=legend, loc="upper left")

    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_network(apr_may_g, title="Apr–May", weight_percentile=50)
plot_network(may_jun_g, title="May–Jun", weight_percentile=50)
plot_network(jun_jul_g, title="Jun–Jul", weight_percentile=50)

In [ ]:
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 150)
non_R6 = meta_2017[meta_2017["plant_type"] == "non-R6"]["field_id"]
R6 = meta_2017[meta_2017["plant_type"] == "R6"]["field_id"]
non_V6 = meta_2017[meta_2017["initial_strain"] == "non-V6"]["field_id"]
V6 = meta_2017[meta_2017["initial_strain"] == "V6"]["field_id"]

labels = jun_jul.index
in_deg = jun_jul.sum(axis=1).to_numpy()
out_deg  = jun_jul.sum(axis=0).to_numpy()

In [ ]:
R6_goes_to = np.concat([non_R6, R6])
non_R6_V6_goes_to = np.concat([non_R6, R6])
non_R6_non_V6_goes_to = np.concat([non_R6])

In [ ]:
def degree_distribution(degrees: np.ndarray, bins = 10):

  y, edges = np.histogram(degrees, bins=bins)
  x = 0.5 * (edges[:-1] + edges[1:])
  plt.loglog(x, y, 'b')
  plt.xlabel("k")
  plt.ylabel("N(k)")
  plt.show()

  pl_fit = ig.statistics.power_law_fit(degrees)
  print(f"The power law exponent α is {pl_fit.alpha}")
  print(f"the power law fit p-value is {pl_fit.p}")
  return pl_fit

In [ ]:
gx = jun_jul_g.to_networkx()
((nx.adjacency_spectrum(gx, weight="weight")))

In [ ]:




if isinstance(gx, nx.classes.digraph.DiGraph):
    indeg_katz_scores = nx.katz_centrality(gx, alpha=0.01)
    outdeg_katz_scores = nx.katz_centrality(gx.reverse(), alpha=0.01)





In [ ]:
indeg_katz_series = pd.Series(indeg_katz_scores, index=list(labels))
outdeg_katz_series = pd.Series(outdeg_katz_scores, index=list(labels))

outdeg_katz_series.value_counts()




In [ ]:
R6_in_pl_fit = degree_distribution(in_deg[R6.index])
R6_out_pl_fit = degree_distribution(out_deg[R6.index])

In [ ]:
R6_V6_in_pl_fit = degree_distribution(in_deg[np.intersect1d(R6.index, V6.index)])
R6_V6_out_pl_fit = degree_distribution(out_deg[np.intersect1d(R6.index, V6.index)])

In [ ]:
non_R6_V6_in_pl_fit = degree_distribution(in_deg[np.intersect1d(non_R6.index, V6.index)])
non_R6_V6_out_pl_fit = degree_distribution(out_deg[np.intersect1d(non_R6.index, V6.index)])

In [ ]:
out_deg[np.intersect1d(non_R6.index, V6.index)]

in_deg[np.intersect1d(non_R6.index, V6.index)]

In [ ]:
np.intersect1d(non_R6.index, V6.index).shape

In [ ]:
R6.index.shape

In [ ]:
degree_distribution(out_deg)